# Версия 3, част 2 — къде са единиците и колко са? (детектор)

Класификаторът от `warcraft3_unit_crops.ipynb` познава изрязана единица, но не знае **къде** са единиците
на целия screenshot и **колко** са. Това е *object detection*: моделът дава по една рамка (box) за всяка
единица, заедно с класа ѝ. Броят на рамките = броят на единиците.

fast.ai няма готов детектор, затова тук използваме **YOLO** от пакета `ultralytics` — също на PyTorch и
със същата идея: зареждаме предварително обучен модел и го *дообучаваме* (transfer learning, като `fine_tune`).

Данни (рамките не се рисуват на ръка, освен при истинските screenshot-и):
- **solo снимки** (`data/single_units`) — един тип, 1–3 копия на известни места;
- **смесени сцени** (`data/units.csv`) — `dataset_trigger.j` записва мястото на всяка единица:
  тълпи, армии, битки, герои, заклинания;
- **битки между компютърни играчи** (`data/battle_units.csv`) — `dataset_trigger_battle.j`: истинска
  melee игра, камерата следи битките и записва всяка видима единица (вкл. височината на терена `z`);
- **истински screenshot-и от игри** (`my_screenshots/train`, `my_screenshots/test`) — рамките са
  проверени на ръка (виж README). `real-test` е честната проверка за целта 98–99%.

Validation и test са **цели карти**, които моделът не вижда при обучението.
Работи локално и в Kaggle (GPU T4 ×2, Internet включен, прикачен dataset от `make_kaggle_dataset.py`).

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
from PIL import Image, ImageDraw
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, shutil, torch

# where the data is (read) and where results go (write): the project folder, or Kaggle's folders
if Path('/kaggle/input').is_dir():
    ROOT = next(Path('/kaggle/input').rglob('single_units/labels.csv')).parents[2]
    WORK = Path('/kaggle/working')
else:
    ROOT = WORK = Path('.')
YOLO_DATA = WORK/'yolo'

MODEL = 'yolo26m.pt'       # m = medium (20M parameters); 'yolo26s.pt' is 2x faster
EPOCHS = 60
HOURS = 10                 # stop in time on Kaggle (12 h limit); YOLO keeps the best epoch
GPUS = list(range(torch.cuda.device_count())) or 'cpu'
ROOT, WORK, GPUS

## Рамка на единица по позицията ѝ

`screen_box(x, y, zoom)`: x, y = мястото на единицата спрямо точката, над която е камерата (в единици на
картата), zoom = разстоянието на камерата. F, CX, CY са измерени от снимките (виж crop notebook-а).
Рамката е по-висока нагоре, за да хване и летящите. `solo_positions` пресмята x, y на копията точно като
`dataset_trigger_single.j`. Единици под долния панел или извън екрана не се броят (не се виждат).

In [ ]:
F, CX, CY = 908, 630, 286
COS, SIN = np.cos(np.radians(56)), np.sin(np.radians(56))

def screen_box(x, y, zoom, z=0):
    "(left, top, right, bottom) in pixels of a 1280x720 screenshot; z = ground height above the camera target"
    depth = zoom + y * COS - z * SIN
    sx, sy, scale = CX + F * x / depth, CY - F * (y * SIN + z * COS) / depth, F / depth
    return (sx - 175*scale, sy - 320*scale, sx + 175*scale, sy + 145*scale)

def visible(box):
    "Is the unit on the battlefield (not off screen, not under the bottom panel)?"
    sx, sy = (box[0] + box[2]) / 2, box[3] - 145 * (box[2] - box[0]) / 350
    return 0 <= sx <= 1280 and 30 <= sy <= 720 and not (sy > 540 and 140 < sx < 1140)

def solo_positions(r):
    offsetx = (r.position % 3 - 1) * 220.0          # same formulas as in dataset_trigger_single.j
    offsety = (r.position // 3 - 1) * 140.0 - 70.0
    return [(offsetx + (i - (r.copies - 1) / 2) * 340, offsety) for i in range(r.copies)]

def hide_worker_icon(img):
    "The idle-worker portrait (bottom left) hints at worker classes; hide it everywhere, also later in the app."
    img = img.copy()
    img.paste((0, 0, 0), (160, 465, 220, 525))
    return img

def split_of(map_name):
    "Maps named ...valid... / ...test... are never trained on; ...trial... maps (short test runs) are not used."
    return 'trial' if 'trial' in map_name else 'valid' if 'valid' in map_name else 'test' if 'test' in map_name else 'train'

## Всички единици в една таблица

Един ред = една видима единица: снимка, име и рамка. Solo и смесените снимки дават рамката от позицията;
истинските screenshot-и — от проверените `.txt` файлове (формат на YOLO, като част от размера на снимката).

In [ ]:
solo = pd.read_csv(ROOT/'data/single_units/labels.csv')
excluded = pd.read_csv(ROOT/'data/single_units/excluded.csv')
# frames excluded only because of the "No clear capture area" text are fine here: the text isn't a unit
warning = set(excluded.loc[excluded['reason'].str.contains('warning text'), 'fname'])
solo = solo[solo['split'].isin(['train', 'valid', 'test']) & ~solo['fname'].isin(set(excluded['fname']) - warning)]
vocab = sorted(solo['labels'].unique())

units, images = [], []      # units: one row per unit; images: one row per screenshot
for r in solo.itertuples():
    path = ROOT/'data/single_units/screens'/r.fname
    images.append((path, r.split, 'solo', r.map))
    for x, y in solo_positions(r):
        units.append((path, r.labels, *screen_box(x, y, r.zoom)))

# generated mixed scenes (dataset_trigger.j) and computer-vs-computer games (dataset_trigger_battle.j)
for csv_name, source in [('units.csv', 'mixed'), ('battle_units.csv', 'battle')]:
    if not (ROOT/'data'/csv_name).is_file(): continue
    for fname, rows in pd.read_csv(ROOT/'data'/csv_name).groupby('fname'):
        path = ROOT/'data/screens'/fname
        map_name = fname.rsplit('_', 1)[0]
        if split_of(map_name) == 'trial': continue
        images.append((path, split_of(map_name), source, map_name))
        for r in rows.itertuples():
            box = screen_box(r.x, r.y, r.zoom, getattr(r, 'z', 0))
            if visible(box): units.append((path, r.label, *box))

for split in ['train', 'test']:
    folder = ROOT/'my_screenshots'/split
    for txt in sorted((folder/'labels').glob('*.txt')) if (folder/'labels').is_dir() else []:
        path = next(p for p in folder.iterdir() if p.stem == txt.stem and p.suffix.lower() in ('.png', '.jpg', '.jpeg'))
        w, h = Image.open(path).size
        images.append((path, 'real-' + split, 'real', 'real'))
        for line in txt.read_text().splitlines():
            c, x, y, bw, bh = [float(v) for v in line.split()]
            units.append((path, vocab[int(c)], (x - bw/2) * w, (y - bh/2) * h, (x + bw/2) * w, (y + bh/2) * h))

units = pd.DataFrame(units, columns=['image', 'label', 'left', 'top', 'right', 'bottom'])
images = pd.DataFrame(images, columns=['image', 'split', 'source', 'map'])
by_image = {path: rows for path, rows in units.groupby('image')}
assert set(units['label']) <= set(vocab)
images.groupby(['source', 'split']).size()

## Данни във формата на YOLO

YOLO иска папки `images/<split>` и `labels/<split>`: за всяка снимка `.txt` с по един ред на единица —
`клас x_център y_център ширина височина` (0…1). Всички снимки се смаляват до 1280 пиксела ширина и
иконата на работниците се скрива. Истинските `train` screenshot-и се копират 4 пъти — малко са, но са
най-близо до целта.

In [ ]:
shutil.rmtree(YOLO_DATA, ignore_errors=True)
folder = {'train': 'train', 'valid': 'val', 'test': 'test', 'real-train': 'train', 'real-test': 'real_test'}
for img_row in images.itertuples():
    split = folder[img_row.split]
    (YOLO_DATA/'images'/split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA/'labels'/split).mkdir(parents=True, exist_ok=True)
    img = Image.open(img_row.image).convert('RGB')
    w, h = img.size
    img = hide_worker_icon(img.resize((1280, round(1280 * h / w))))
    lines = []
    rows = by_image.get(img_row.image, units.iloc[:0])
    for u in rows.itertuples():
        l, t, r, b = max(u.left, 0), max(u.top, 0), min(u.right, w), min(u.bottom, h)
        lines.append(f'{vocab.index(u.label)} {(l+r)/2/w:.5f} {(t+b)/2/h:.5f} {(r-l)/w:.5f} {(b-t)/h:.5f}')
    copies = 4 if img_row.split == 'real-train' else 1
    for k in range(copies):
        name = f'{img_row.source}_{Path(img_row.image).stem}' + (f'_{k}' if copies > 1 else '')
        img.save(YOLO_DATA/'images'/split/f'{name}.jpg', quality=95)
        (YOLO_DATA/'labels'/split/f'{name}.txt').write_text('\n'.join(lines))

yaml = f"path: {YOLO_DATA.resolve().as_posix()}\ntrain: images/train\nval: images/val\nnames:\n"
yaml += ''.join(f'  {i}: {name}\n' for i, name in enumerate(vocab))
(YOLO_DATA/'data.yaml').write_text(yaml)
{split.name: len(list(split.iterdir())) for split in sorted((YOLO_DATA/'images').iterdir())}

In [ ]:
# check: draw the boxes of a few training images (mixed scenes first, if there are any)
names = list(images.loc[images['split'].eq('train') & images['source'].eq('mixed'), 'image'][:2])
names += list(images.loc[images['split'].eq('train'), 'image'].sample(4 - len(names), random_state=0))
sheet = Image.new('RGB', (1280, 720))
for k, path in enumerate(names):
    img = Image.open(path).convert('RGB')
    draw = ImageDraw.Draw(img)
    for u in by_image.get(path, units.iloc[:0]).itertuples():
        draw.rectangle((u.left, u.top, u.right, u.bottom), outline=(255, 0, 255), width=4)
        draw.text((u.left + 4, u.top + 4), u.label, fill=(255, 255, 0))
    sheet.paste(img.resize((640, 360)), ((k % 2) * 640, (k // 2) * 360))
sheet

## Обучение и аугментация

- `imgsz=1280` — пълната ширина: единиците са малки, не ги смаляваме.
- **mosaic** (вградено в YOLO) слепва 4 снимки в една — още повече смесени сцени.
- огледален образ (`fliplr=0.5`), без обръщане надолу и без въртене: камерата е винаги под същия ъгъл.
- `scale=0.3` — ±30% размер, като различния zoom; `hsv_v=0.4` — ден/нощ.
- `time=HOURS` спира навреме; YOLO пази най-добрата епоха (`best.pt`) по validation.
- Ако обучението спре по средата, продължава с
  `YOLO(WORK/'models/detector/weights/last.pt').train(resume=True)`.

In [ ]:
model = YOLO(MODEL)
model.train(data=str(YOLO_DATA/'data.yaml'), imgsz=1280, epochs=EPOCHS, time=HOURS,
            batch=4 * len(GPUS) if GPUS != 'cpu' else 2, device=GPUS, workers=4,
            fliplr=0.5, flipud=0.0, degrees=0, mosaic=1.0, scale=0.3, hsv_v=0.4,
            project=str((WORK/'models').resolve()), name='detector', exist_ok=True, seed=42, plots=True)

In [ ]:
pd.read_csv(WORK/'models/detector/results.csv')[['epoch', 'metrics/precision(B)', 'metrics/recall(B)',
                                                  'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(10)

## Детектор + класификатор

1. Понякога YOLO дава **две рамки на една единица**. Ако две рамки се застъпват повече от `iou`, пазим
   само по-сигурната. При тълпи и истинските единици се застъпват, затова `iou` се избира по validation
   (1.01 = не махаме нищо).
2. Всяка рамка има същата форма като crop-овете от `warcraft3_unit_crops.ipynb`, затова можем да
   попитаме и fast.ai класификатора коя е единицата. Дали това помага, решава validation: в тълпа
   crop-ът съдържа няколко единици (на test crop-овете класификаторът позна 84.1%), а YOLO вижда цялата
   сцена и знае коя рамка коя единица е.

In [ ]:
from fastai.vision.all import load_learner, PILImage
best = YOLO(WORK/'models/detector/weights/best.pt')
pkl = WORK/'unit_crops/unit_classifier.pkl'
if not pkl.is_file(): pkl = next(Path('/kaggle/input').rglob('unit_classifier.pkl'))  # output of the crop notebook
classifier = load_learner(pkl)

def iou(a, b):
    w = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return w * h / ((a[2]-a[0]) * (a[3]-a[1]) + (b[2]-b[0]) * (b[3]-b[1]) - w * h)

def detect(path):
    "Every box YOLO finds (confidence >= 0.1): (box, YOLO's name, confidence, classifier's name), most confident first."
    img = Image.open(path).convert('RGB')
    img = img.resize((1280, round(1280 * img.height / img.width)))
    r = best.predict(hide_worker_icon(img), imgsz=1280, conf=0.1, verbose=False)[0]
    boxes = r.boxes.xyxy.tolist()
    crops = [PILImage.create(np.array(img.crop(tuple(round(v) for v in b)))) for b in boxes]
    names = []
    if crops:
        with classifier.no_bar(): probs, _ = classifier.get_preds(dl=classifier.dls.test_dl(crops))
        names = [classifier.dls.vocab[k] for k in probs.argmax(1).tolist()]
    found = zip(boxes, [vocab[int(c)] for c in r.boxes.cls.tolist()], r.boxes.conf.tolist(), names)
    return sorted(found, key=lambda f: -f[2])

def find_units(found, conf, max_iou, use_classifier=True):
    "The list of unit names on a screenshot, one per unit."
    keep = []
    for f in found:
        if f[2] >= conf and all(iou(f[0], k[0]) < max_iou for k in keep): keep.append(f)
    return sorted(k[3] if use_classifier else k[1] for k in keep)

# the true list of units on every screenshot
truth = {path: sorted(rows['label']) for path, rows in by_image.items()}
detections = {path: detect(path) for path in images.loc[images['split'] != 'train', 'image']}

def score(split, conf, max_iou, use_classifier=True):
    "Share of screenshots where the list of units (with repeats) is exactly right."
    paths = images.loc[images['split'] == split, 'image']
    return np.mean([find_units(detections[p], conf, max_iou, use_classifier) == truth.get(p, []) for p in paths])

## Избор на праговете по validation

Пробваме няколко прага за увереност (`conf`) и застъпване (`iou`) — само на validation картите.

In [ ]:
grid = pd.DataFrame([(c, m, score('valid', c, m, False), score('valid', c, m))
                     for c in [0.2, 0.3, 0.4, 0.5] for m in [0.5, 0.7, 1.01]],
                    columns=['conf', 'iou', 'YOLO only', 'YOLO + classifier'])
USE_CLASSIFIER = bool(grid['YOLO + classifier'].max() > grid['YOLO only'].max())   # only if it really helps
best_row = grid.loc[grid['YOLO + classifier' if USE_CLASSIFIER else 'YOLO only'].idxmax()]
CONF, MAX_IOU = float(best_row['conf']), float(best_row['iou'])
print('chosen conf', CONF, 'iou', MAX_IOU, 'use classifier', USE_CLASSIFIER)
grid

## Резултат: изцяло вярна снимка

„Изцяло вярна“ = същите единици **и** същият брой от всяка. Test картите и истинските screenshot-и
(`real-test`) се гледат **веднъж**, след избора на праговете. Ако после променяме нещо заради тях, те вече
не са честна проверка.

In [ ]:
results = pd.DataFrame([(split, (images['split'] == split).sum(), score(split, CONF, MAX_IOU, False), score(split, CONF, MAX_IOU))
                        for split in ['valid', 'test', 'real-test'] if (images['split'] == split).any()],
                       columns=['split', 'screenshots', 'YOLO only', 'YOLO + classifier'])
results

In [ ]:
# what goes wrong on the honest tests
wrong = []
for split in ['test', 'real-test']:
    for p in images.loc[images['split'] == split, 'image']:
        guess, true = find_units(detections[p], CONF, MAX_IOU, USE_CLASSIFIER), truth.get(p, [])
        if guess != true:
            wrong.append((split, Path(p).name, len(true), list((Counter(true) - Counter(guess)).elements()),
                          list((Counter(guess) - Counter(true)).elements())))
wrong = pd.DataFrame(wrong, columns=['split', 'image', 'units', 'missed', 'named but not there'])
print(len(wrong), 'wrong screenshots')
wrong.head(30)

In [ ]:
# old Version 2 map4 (names only, no counts): the same "whole image correct" measure as Version 2 (18%)
if (ROOT/'data/labels.csv').is_file():
    old = pd.read_csv(ROOT/'data/labels.csv')
    old = old[old['fname'].str.startswith('map4_')]
    ok = [set(find_units(detect(ROOT/'data/screens'/f), CONF, MAX_IOU, USE_CLASSIFIER)) == set(l.split(';'))
          for f, l in zip(old['fname'], old['labels'])]
    if ok: print(f'map4 (Version 2 validation map): whole image correct {np.mean(ok):.1%} of {len(ok)}')

## Export за сайта

ONNX файлът работи в браузъра с onnxruntime-web, както моделът от Версия 2. Праговете се записват до него.

In [ ]:
best.export(format='onnx', imgsz=1280)
(WORK/'models/detector/thresholds.json').write_text(
    f'{{"conf": {CONF}, "iou": {MAX_IOU}, "use_classifier": {str(USE_CLASSIFIER).lower()}}}')
# on Kaggle everything in /kaggle/working becomes the notebook's output: keep the models, not the images
if WORK != ROOT: shutil.rmtree(YOLO_DATA)